In [57]:
"""
Remove book quotes from GoodReads reviews of "All Systems Red"

This script extracts text from the book PDF, identifies quoted passages in reviews,
and removes them to clean the review dataset.
"""

import pandas as pd
import re
from pathlib import Path
import PyPDF2
from difflib import SequenceMatcher
import warnings
warnings.filterwarnings('ignore')

# Configuration
BOOK_PDF_PATH = "/Users/ishanicheshire/Desktop/murderbot/murderbot_test/MB_text/All_Systems_Red.pdf"
REVIEWS_CSV_PATH = "/Users/ishanicheshire/Desktop/murderbot/murderbot_test/analysis/man_cleaned/final_all_id.csv"
OUTPUT_CSV_PATH = "/Users/ishanicheshire/Desktop/murderbot/murderbot_test/analysis/man_cleaned/final_all_id_cleaned.csv"

# Minimum number of consecutive words to consider as a quote
MIN_QUOTE_WORDS = 5
# Similarity threshold (0-1) for fuzzy matching
SIMILARITY_THRESHOLD = 0.85


def extract_book_text(pdf_path):
    """Extract text from the book PDF."""
    print("Extracting text from book PDF...")
    
    book_text = []
    
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            total_pages = len(pdf_reader.pages)
            
            for page_num in range(total_pages):
                page = pdf_reader.pages[page_num]
                text = page.extract_text()
                if text:
                    book_text.append(text)
                
                if (page_num + 1) % 10 == 0:
                    print(f"  Processed {page_num + 1}/{total_pages} pages")
        
        full_text = " ".join(book_text)
        # Clean up the text
        full_text = re.sub(r'\s+', ' ', full_text).strip()
        
        print(f"Extracted {len(full_text)} characters from {total_pages} pages")
        return full_text
    
    except Exception as e:
        print(f"Error extracting PDF text: {e}")
        return ""


def create_ngrams(text, n):
    """Create n-grams from text."""
    words = text.split()
    ngrams = []
    for i in range(len(words) - n + 1):
        ngram = ' '.join(words[i:i + n])
        ngrams.append((ngram, i))
    return ngrams


def normalize_text(text):
    """Normalize text for comparison."""
    # Convert to lowercase
    text = text.lower()
    # Remove common punctuation variations
    text = re.sub(r'[""'']', '"', text)
    text = re.sub(r'[–—]', '-', text)
    
    # Normalize common British/American spelling differences
    # This helps match quotes where reviewers use different spellings
    spelling_variations = {
        'armour': 'armor',
        'colour': 'color',
        'honour': 'honor',
        'favour': 'favor',
        'labour': 'labor',
        'neighbour': 'neighbor',
        'rumour': 'rumor',
        'humour': 'humor',
        'behaviour': 'behavior',
    }
    
    for british, american in spelling_variations.items():
        text = re.sub(r'\b' + british + r'\b', american, text)
    
    # Remove most punctuation but keep spaces (helps match across punctuation differences)
    text = re.sub(r'[^\w\s]', ' ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def extract_quoted_sections(text):
    """Extract all quoted sections from text along with their positions."""
    quoted_sections = []
    
    # Define quote pairs (opening, closing)
    quote_pairs = [
        ('"', '"'),      # Standard double quotes
        ('"', '"'),      # Curly double quotes
        ("'", "'"),      # Curly single quotes (NOT standard apostrophes)
    ]
    
    for open_q, close_q in quote_pairs:
        i = 0
        while i < len(text):
            # Find opening quote
            start = text.find(open_q, i)
            if start == -1:
                break
            
            # For standard single quotes, check if this is actually an apostrophe
            if open_q == "'" and start > 0:
                prev_char = text[start - 1]
                next_char = text[start + 1] if start + 1 < len(text) else ''
                if prev_char.isalpha() or next_char.isalpha():
                    i = start + 1
                    continue
            
            # Find the nearest closing quote
            end = text.find(close_q, start + 1)
            if end == -1:
                i = start + 1
                continue
            
            # Extract the quoted text
            quoted_text = text[start + 1:end].strip()
            word_count = len(quoted_text.split())
            
            # Skip if too short (< 5 words)
            if word_count < 5:
                # BUT: if this short quote ends with sentence punctuation and is followed by
                # non-quote text, it's probably complete - move past it
                if end > 0 and text[end-1] in '.!?' and end + 1 < len(text):
                    after_char = text[end + 1] if end + 1 < len(text) else ''
                    # If next char is not a quote, skip this short complete quote
                    if after_char not in ['"', '"', "'", "'"]:
                        i = end + 1
                        continue
                i = start + 1
                continue
            
            # Skip if starts with ) or . (grabbed text after quote)
            if quoted_text.startswith(')') or quoted_text.startswith('.'):
                i = start + 1
                continue
            
            # Check if this looks like an embedded phrase vs a standalone quote
            # If it ends with . ! or ? and there's lowercase text immediately after the close quote,
            # it's probably a short embedded phrase, not a book quote we want
            if word_count <= 15 and end > 0:
                last_char_in_quote = text[end - 1]
                if last_char_in_quote in '.!?':
                    # Check what comes after the closing quote
                    after_quote = text[end + 1:end + 20].strip() if end + 1 < len(text) else ''
                    # If it continues with lowercase (e.g., "my humans." The mystery...)
                    # this is probably embedded, not a standalone quote
                    if after_quote and after_quote[0].isupper() and not after_quote[0].isdigit():
                        # Starts with capital - could be start of new sentence after quote
                        # This is likely a short embedded phrase, skip it
                        if word_count <= 3:  # Very short phrases like "my humans" are not book quotes
                            i = end + 1
                            continue
            
            # Valid quote
            quoted_sections.append({
                'text': quoted_text,
                'start': start,
                'end': end + 1,
                'word_count': word_count
            })
            
            # Move past this quote
            i = end + 1
    
    # Sort by position
    quoted_sections.sort(key=lambda x: x['start'])
    
    # Remove overlapping quotes (keep the first one)
    filtered = []
    for quote in quoted_sections:
        overlaps = False
        for existing in filtered:
            if (quote['start'] < existing['end'] and quote['end'] > existing['start']):
                overlaps = True
                break
        if not overlaps:
            filtered.append(quote)
    
    return filtered


def is_text_from_book(text, book_text, min_words=5, debug=False):
    """Check if a text passage appears in the book."""
    if len(text.split()) < min_words:
        return False
    
    norm_text = normalize_text(text)
    norm_book = normalize_text(book_text)
    
    if debug:
        print(f"\n  Checking quote: '{text[:60]}...'")
        print(f"  Normalized: '{norm_text[:60]}...'")
    
    # Check for exact match first
    if norm_text in norm_book:
        if debug: print(f"  ✓ Exact match found")
        return True
    
    # For quotes with ellipses, the reviewer is indicating they skipped text
    # Split on ellipses and check if each part exists in the book
    if '..' in text or '…' in text:
        # Split on any ellipses pattern (2+ dots or ellipsis character)
        parts = re.split(r'\.{2,}|…+', text)
        parts = [normalize_text(p.strip()) for p in parts if len(p.strip().split()) >= 3]
        
        if debug:
            print(f"  Found {len(parts)} parts separated by ellipses")
        
        if len(parts) >= 1:
            # Check each part individually - they should all be in the book
            parts_in_book = []
            for i, part in enumerate(parts):
                in_book = part in norm_book
                parts_in_book.append(in_book)
                if debug:
                    print(f"    Part {i+1}: '{part[:40]}...' -> {'✓' if in_book else '✗'}")
            
            # If at least 80% of parts are found, it's a book quote
            matches = sum(parts_in_book)
            if matches >= len(parts) * 0.8:
                if debug: print(f"  ✓ Ellipses-based match: {matches}/{len(parts)} parts found")
                return True
    
    # If no ellipses, continue with regular matching
    words = norm_text.split()
    
    # For quotes with at least 8 words, check if the first 60% appears
    if len(words) >= 8:
        partial_len = max(5, int(len(words) * 0.6))
        partial_quote = ' '.join(words[:partial_len])
        if partial_quote in norm_book:
            if debug: print(f"  ✓ Partial match (first 60%): '{partial_quote[:40]}...'")
            return True
    
    # Try checking chunks of the quote - be aggressive
    if len(words) < min_words:
        return False
    
    # For quotes of different lengths, use different strategies
    if len(words) <= 12:
        # For short quotes, require 60% continuous match
        threshold_words = max(min_words, int(len(words) * 0.6))
    else:
        # For longer quotes, look for 8+ word sequences
        threshold_words = max(8, int(len(words) * 0.4))
    
    # Check if we can find a continuous sequence of threshold length
    for length in range(len(words), threshold_words - 1, -1):
        for i in range(len(words) - length + 1):
            sequence = ' '.join(words[i:i + length])
            if sequence in norm_book:
                if debug: print(f"  ✓ Continuous sequence match: {length}/{len(words)} words")
                return True
    
    # Even more aggressive: check if 30% continuous match exists anywhere
    threshold_words_aggressive = max(5, int(len(words) * 0.3))
    for length in range(len(words), threshold_words_aggressive - 1, -1):
        for i in range(len(words) - length + 1):
            sequence = ' '.join(words[i:i + length])
            if sequence in norm_book:
                if debug: print(f"  ✓ Aggressive match: {length}/{len(words)} words (30% threshold)")
                return True
    
    # Last resort: for any quote, check if we can find at least 2 distinct chunks of 5+ words
    if len(words) >= 10:
        chunks_found = 0
        chunk_size = 5
        for i in range(0, len(words) - chunk_size + 1, chunk_size):
            chunk = ' '.join(words[i:i + chunk_size])
            if chunk in norm_book:
                chunks_found += 1
                if chunks_found >= 2:
                    if debug: print(f"  ✓ Multiple chunks match: {chunks_found}+ chunks of 5 words")
                    return True
    
    if debug:
        print(f"  ✗ No match found")
        print(f"  Looking for: '{norm_text[:80]}...'")
        # Try to find any 5-word sequence that matches
        for i in range(min(5, len(words) - 5)):
            seq = ' '.join(words[i:i+5])
            if seq in norm_book:
                print(f"  Found 5-word match at position {i}: '{seq}'")
                break
    
    return False


def find_book_quotes_v2(review_text, book_text, min_words=MIN_QUOTE_WORDS, debug=False):
    """
    Find quoted sections in the review that come from the book.
    Only looks at text within quotation marks.
    Returns list of quote dictionaries with position info.
    """
    if not review_text or pd.isna(review_text):
        return []
    
    # Extract all explicitly quoted sections
    quoted_sections = extract_quoted_sections(review_text)
    
    if debug:
        print(f"\nFound {len(quoted_sections)} quoted sections:")
        for i, q in enumerate(quoted_sections, 1):
            print(f"  {i}. '{q['text'][:50]}...' ({len(q['text'].split())} words)")
    
    # Check which quotes are from the book
    book_quotes = []
    for quote in quoted_sections:
        is_from_book = is_text_from_book(quote['text'], book_text, min_words, debug=debug)
        if is_from_book:
            book_quotes.append(quote)
    
    if debug:
        print(f"\nTotal book quotes found: {len(book_quotes)}")
    
    return book_quotes


def remove_quotes_from_review_v2(review_text, book_quotes):
    """Remove book quotes from the review text."""
    if not review_text or pd.isna(review_text) or not book_quotes:
        return review_text
    
    # Sort by position in reverse to process from end to start
    book_quotes = sorted(book_quotes, key=lambda x: x['start'], reverse=True)
    
    cleaned_text = review_text
    
    for quote in book_quotes:
        start = quote['start']
        end = quote['end']
        
        # Simply replace the quoted section (including the quotation marks) with [quoted text]
        cleaned_text = cleaned_text[:start] + '[quoted text]' + cleaned_text[end:]
    
    # Clean up multiple consecutive markers
    cleaned_text = re.sub(r'(\[quoted text\]\s*){2,}', '[quoted text] ', cleaned_text)
    
    # Clean up extra whitespace
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text


def process_reviews(reviews_df, book_text):
    """Process all reviews to remove book quotes."""
    print(f"\nProcessing {len(reviews_df)} reviews...")
    
    cleaned_reviews = []
    quotes_found = 0
    examples_for_display = []  # Store examples for later display
    
    for idx, row in reviews_df.iterrows():
        review_text = row['review_text']
        
        # Enable debug for first few reviews with quotes
        debug = (quotes_found < 3)
        
        # Find quotes in this review
        book_quotes = find_book_quotes_v2(review_text, book_text, debug=debug)
        
        if book_quotes:
            quotes_found += 1
            cleaned_text = remove_quotes_from_review_v2(review_text, book_quotes)
            cleaned_reviews.append(cleaned_text)
            
            # Store first 20 examples
            if len(examples_for_display) < 20:
                examples_for_display.append({
                    'index': idx,
                    'original': review_text,
                    'cleaned': cleaned_text,
                    'num_quotes': len(book_quotes)
                })
        else:
            cleaned_reviews.append(review_text)
        
        if (idx + 1) % 500 == 0:
            print(f"  Processed {idx + 1}/{len(reviews_df)} reviews ({quotes_found} with quotes)")
    
    print(f"\nTotal reviews with quotes found: {quotes_found}/{len(reviews_df)}")
    
    return cleaned_reviews, examples_for_display


def display_comparison_examples(examples):
    """Display side-by-side comparison of original and cleaned reviews."""
    print("\n" + "="*100)
    print("SIDE-BY-SIDE COMPARISON OF FIRST 20 ALTERED REVIEWS")
    print("="*100)
    
    for i, example in enumerate(examples, 1):
        print(f"\n{'─'*100}")
        print(f"EXAMPLE {i} (Review Index: {example['index']}, Quotes Found: {example['num_quotes']})")
        print(f"{'─'*100}")
        
        original = example['original']
        cleaned = example['cleaned']
        
        print("\n📄 ORIGINAL REVIEW:")
        print("─" * 100)
        # Wrap text for better readability
        print(wrap_text(original, width=95))
        
        print("\n✨ CLEANED REVIEW:")
        print("─" * 100)
        print(wrap_text(cleaned, width=95))
        
        # Show statistics
        orig_len = len(original)
        clean_len = len(cleaned)
        removed = orig_len - clean_len
        percent_removed = (removed / orig_len * 100) if orig_len > 0 else 0
        
        print("\n📊 STATISTICS:")
        print(f"   Original length: {orig_len:,} characters")
        print(f"   Cleaned length:  {clean_len:,} characters")
        print(f"   Removed:         {removed:,} characters ({percent_removed:.1f}%)")
        
        if i < len(examples):
            print()  # Extra line between examples


def wrap_text(text, width=95):
    """Wrap text to specified width for display."""
    words = text.split()
    lines = []
    current_line = []
    current_length = 0
    
    for word in words:
        word_length = len(word) + 1  # +1 for space
        if current_length + word_length > width and current_line:
            lines.append(' '.join(current_line))
            current_line = [word]
            current_length = word_length
        else:
            current_line.append(word)
            current_length += word_length
    
    if current_line:
        lines.append(' '.join(current_line))
    
    return '\n'.join(lines)



In [58]:
# raise Exception()

In [59]:
"""Main execution function."""
print("="*60)
print("Book Quote Removal Script")
print("="*60)

# Check if files exist
if not Path(BOOK_PDF_PATH).exists():
    print(f"Error: Book PDF not found at {BOOK_PDF_PATH}")
    raise Exception()

if not Path(REVIEWS_CSV_PATH).exists():
    print(f"Error: Reviews CSV not found at {REVIEWS_CSV_PATH}")
    raise Exception()

# Extract book text
book_text = extract_book_text(BOOK_PDF_PATH)

if not book_text:
    print("Error: Could not extract text from book PDF")
    raise Exception()

# Load reviews
print(f"\nLoading reviews from CSV...")
reviews_df = pd.read_csv(REVIEWS_CSV_PATH)
print(f"Loaded {len(reviews_df)} reviews")

if 'review_text' not in reviews_df.columns:
    print("Error: 'review_text' column not found in CSV")
    print(f"Available columns: {list(reviews_df.columns)}")
    raise Exception()

# Process reviews
cleaned_reviews, examples = process_reviews(reviews_df, book_text)

# Display side-by-side comparisons
if examples:
    display_comparison_examples(examples)

# Add cleaned reviews to dataframe
reviews_df['review_text_cleaned'] = cleaned_reviews

# Save results
print(f"\nSaving cleaned reviews to {OUTPUT_CSV_PATH}...")
reviews_df.to_csv(OUTPUT_CSV_PATH, index=False)
print("Done!")

# Print statistics
original_lengths = reviews_df['review_text'].str.len()
cleaned_lengths = reviews_df['review_text_cleaned'].str.len()
chars_removed = (original_lengths - cleaned_lengths).sum()

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total reviews processed: {len(reviews_df)}")
print(f"Total characters removed: {chars_removed:,}")
print(f"Average characters removed per review: {chars_removed/len(reviews_df):.1f}")
print(f"Output saved to: {OUTPUT_CSV_PATH}")

Book Quote Removal Script
Extracting text from book PDF...
  Processed 10/92 pages
  Processed 20/92 pages
  Processed 30/92 pages
  Processed 40/92 pages
  Processed 50/92 pages
  Processed 60/92 pages
  Processed 70/92 pages
  Processed 80/92 pages
  Processed 90/92 pages
Extracted 178860 characters from 92 pages

Loading reviews from CSV...
Loaded 3869 reviews

Processing 3869 reviews...

Found 0 quoted sections:

Total book quotes found: 0

Found 0 quoted sections:

Total book quotes found: 0

Found 0 quoted sections:

Total book quotes found: 0

Found 3 quoted sections:
  1. 'I could have become a mass murderer after I hacked...' (31 words)
  2. 'So, I'm awkward with actual humans....I know I'm a...' (28 words)
  3. 'I got my helmet on and opaqued it. The relief was ...' (21 words)

  Checking quote: 'I could have become a mass murderer after I hacked my govern...'
  Normalized: 'i could have become a mass murderer after i hacked my govern...'
  ✓ Continuous sequence match: 27/31 

In [20]:
# reviews_df.head()